# 6. Fit explicit network weights through equilibrium

**Goal:** combine lesson 4's equilibrium differentiation with lesson 5's tiny network.
Only the two output weights are optimized. Hidden weights, geometry, load, and the
analytic baseline stay fixed. No pretrained checkpoint or earlier notebook output
is required.

The path is now **weights → density → native equilibrium → displacement loss**.
The `theta` slot will contain neural weights, not Lamé parameters.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
from litepgo import dof, energy, fem, solver
from litepgo.torch import TorchVolumetricMaterial, response
from common import make_geometry

torch.set_num_threads(1)
dtype = torch.float64

## Recreate the same density, then expose the output weights

The analytic law and tiny network are repeated here so the notebook is self-contained.
We will pass each row of `theta` straight into the elementwise dot product. There is
no temporary module modification, lock, or hidden parameter update.

In [ ]:
class TorchNeoHookean(TorchVolumetricMaterial):
    num_parameters = 2

    @staticmethod
    def W(F, Fp, theta):
        Fe = F @ torch.linalg.inv(Fp)
        lam, mu = theta.unbind(-1)
        log_J = torch.linalg.det(Fe).log()
        psi = 0.5 * mu * (Fe.square().sum((-2, -1)) - 3) - mu * log_J + 0.5 * lam * log_J.square()
        W = torch.linalg.det(Fp) * psi
        return torch.where((lam >= 0) & (mu > 0), W, torch.full_like(W, float('nan')))

A = torch.tensor([[1.0, 0.5], [-0.7, 1.2]], dtype=dtype)
b = torch.tensor([0.2, -0.1], dtype=dtype)
alpha = 0.2

def invariants(Fe):
    J = torch.linalg.det(Fe)
    I1 = Fe.square().sum((-2, -1))
    return torch.stack((J.pow(-2 / 3) * I1 - 3, J.log()), dim=-1)

def neural_density(F, Fp, weights):
    Fe = F @ torch.linalg.inv(Fp)
    z = invariants(Fe)
    hidden = torch.nn.functional.softplus(z @ A.T + b)
    network = (hidden * weights).sum(-1)
    correction = alpha * z.square().sum(-1) * network
    physical = F.new_tensor([2., 3.]).expand(len(F), 2)
    return TorchNeoHookean.W(F, Fp, physical) + torch.linalg.det(Fp) * correction

class TrainableNeuralMaterial(TorchVolumetricMaterial):
    num_parameters = 2

    @staticmethod
    def W(F, Fp, theta):
        return neural_density(F, Fp, theta)

X, topology, mapping, geometry, plastic = make_geometry()
key = (TrainableNeuralMaterial, fem.TetP1Basis, fem.TetCentroidQuadrature)
truth = np.array([0.6, 0.4])

## Local independence and global sharing

For a batch of $N$ points, `theta` has shape `(N,2)`. Each row is independent at the
material interface, so the adapter can differentiate `W.sum()` safely.
`ConstantDofMap(topology, 2)` then maps all those rows to the **same global pair** $p$.
Native assembly sums their derivative contributions. Sharing values does not mean
reading `theta[0]` for every row.

First check the mixed derivative $\partial^2W/\partial F\partial\theta$ with distinct
parameter rows. This catches accidentally coupling points before any solver is involved.

In [ ]:
F = np.array([np.diag([1.08, .98, 1.03]), np.diag([.97, 1.06, 1.02])])
Fp = np.repeat(np.eye(3)[None], 2, axis=0)
theta = np.array([[.6, .4], [.8, .3]])
out = TrainableNeuralMaterial.eval(F, Fp, theta, outputs=('dW_dF', 'd2W_dF_dtheta'))
h = 1e-5
for j in range(2):
    delta = np.zeros_like(theta)
    delta[:, j] = h
    plus = TrainableNeuralMaterial.eval(F, Fp, theta + delta, outputs=('dW_dF',))['dW_dF']
    minus = TrainableNeuralMaterial.eval(F, Fp, theta - delta, outputs=('dW_dF',))['dW_dF']
    np.testing.assert_allclose(out['d2W_dF_dtheta'][..., j], (plus-minus)/(2*h), rtol=1e-5, atol=1e-9)
    single = TrainableNeuralMaterial.eval(F[j:j+1], Fp[j:j+1], theta[j:j+1], outputs=('dW_dF',))
    np.testing.assert_allclose(single['dW_dF'][0], out['dW_dF'][j], atol=1e-12)
print('mixed derivatives and independent rows passed')

## Reuse the equilibrium interface unchanged

Use lesson 4's mesh, constraints, and load. The code below differs only in the material
class bound to `key`. The parameter count happens to still be two, but their meaning
has changed. The external load remains independent of the weights.

In [ ]:
free = np.array([9, 10, 11])
fixed = dof.DofReduction(mapping.num_dofs, [*range(9), 12, 13, 14])
initial_u = np.zeros(mapping.num_dofs)
load = np.zeros(mapping.num_dofs)
load[free] = [0.015, -0.01, 0.02]
import scipy.sparse as sp
external = energy.QuadraticEnergy(sp.csc_matrix((mapping.num_dofs, mapping.num_dofs)), -load)
options = solver.NewtonOptions(gradient_tolerance=1e-12, max_iterations=100)

parameter_map = dof.ConstantDofMap(topology, 2)

def make_energy(p):
    fields = [energy.AffineResponse(np.zeros((2, 3)), p)] * topology.num_cells
    elastic = fem.VolumetricFemEnergy[key](mapping, geometry, plastic, fields)
    return energy.WeightedSumEnergy([(elastic, 1.), (external, 1.)])

def make_response(u):
    return fem.VolumetricFemForceMaterialResponse[key](parameter_map, mapping, geometry, u, plastic)

equilibrium = solver.StaticEquilibriumResponse(2, fixed, initial_u, options, make_energy, make_response)

## Optimize an explicit weight tensor

Unlike physical positive moduli, these network weights do not require an exponential
parameter transform. The local starting point is a small perturbation of reference
weights. The baseline helps this example remain stable; arbitrary weight updates are
not guaranteed to be feasible, and solver failures propagate to the optimizer.

In [ ]:
target = torch.from_numpy(equilibrium.eval_value(truth).copy())
p = torch.tensor(truth + [.2, -.1], dtype=dtype, requires_grad=True)
start = torch.from_numpy(equilibrium.eval_value(p.detach().numpy()).copy())
scale = (start[free] - target[free]).square().mean().sqrt().item()
assert scale > 1e-10

def loss(p):
    predicted = response(equilibrium, p)
    return 0.5 * ((predicted[free] - target[free]) / scale).square().mean()

print('initial loss:', loss(p).item())

## Check the entire gradient before fitting

The finite differences below include new equilibrium solves. Agreement checks both
the local mixed derivative and the implicit state dependence. The outer bridge is
first-order: it is not a way to request double backward through the native solver.

In [ ]:
analytic = torch.autograd.grad(loss(p), p)[0]
h = 1e-4
numerical = torch.stack([(loss(p.detach() + d) - loss(p.detach() - d)) / (2 * h)
                         for d in h * torch.eye(len(p), dtype=dtype)])
torch.testing.assert_close(analytic, numerical, rtol=2e-4, atol=2e-6)
print('analytic gradient:', analytic.detach().numpy())
print('finite difference:', numerical.detach().numpy())

## Fit and inspect the observations

The optimizer owns `p`, not `model.parameters()`. No weight copying back into a module
is needed: each evaluation reads the current explicit tensor through the material
parameter field. We require loss reduction, not unique recovery from this small observation.

In [ ]:
history = []
optimizer = torch.optim.LBFGS([p], max_iter=25, tolerance_grad=1e-10,
                            tolerance_change=1e-14, line_search_fn='strong_wolfe')

def closure():
    optimizer.zero_grad()
    value = loss(p)
    value.backward()
    history.append(value.item())
    return value

before = loss(p).item()
optimizer.step(closure)
after = loss(p).item()
assert np.isfinite(after) and after < before * 1e-5
print('loss:', before, '->', after)
print('fitted network weights:', p.detach().numpy())
fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
ax.semilogy(np.maximum(history, 1e-18), marker='.')
ax.set(xlabel='L-BFGS closure evaluation (includes line-search trials)', ylabel='Loss')
ax.grid(alpha=0.25)
plt.show()
print('reference network weights:', truth)
state = equilibrium.eval_value(p.detach().numpy())
assert np.linalg.norm(make_energy(p.detach().numpy()).eval_gradient(state)[free]) < 1e-9
print('target / fitted free displacement:')
print(np.column_stack((target[free].numpy(), state[free])))

## Try it and continue

Change the starting weights slightly and rerun. Change the load direction and regenerate
the target. Try two load cases before concluding that a weight pair is identifiable.

You have now traversed the full chain. The advanced `neural_equilibrium_fit.py` example
uses a larger invariant network and an optimized reference-state correction; the
benchmark measures costs at larger mesh and parameter counts. Full-network training,
GPU execution, shell materials, and plastic fitting are separate next steps—not hidden
requirements for understanding this six-lesson path.